# From Raw Counts to Differential Expression (DESeq2, R)

_BioNexus Hub - bionexushub.com · Companion notebook to the lesson._

**Set the runtime to R first:** Runtime, then Change runtime type, then R. Run each cell with Shift+Enter.


## 1. Set up
Installing Bioconductor packages takes a few minutes the first time.


In [ ]:
if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")
BiocManager::install(c("DESeq2", "apeglm"), update = FALSE, ask = FALSE)
library(DESeq2)


## 2. Load the data
Count matrix (genes x samples) plus a metadata table with a condition column.


In [ ]:
base <- "https://raw.githubusercontent.com/owkin/PyDESeq2/main/datasets/synthetic/"
counts <- as.matrix(read.csv(paste0(base, "test_counts.csv"), row.names = 1))
metadata <- read.csv(paste0(base, "test_metadata.csv"), row.names = 1)
metadata$condition <- factor(metadata$condition)
metadata <- metadata[colnames(counts), , drop = FALSE]
dim(counts)


## 3. Build the dataset, filter, and fit
The design ~condition is the comparison. We also drop near-empty genes and check the coefficient names.


In [ ]:
dds <- DESeqDataSetFromMatrix(countData = counts,
                              colData   = metadata,
                              design    = ~ condition)

# drop genes with almost no reads
keep <- rowSums(counts(dds)) >= 10
dds <- dds[keep, ]

dds <- DESeq(dds)
resultsNames(dds)   # coefficient names (needed for shrinkage below)


## 4. Read the results
Trust the padj column. NA in padj means the gene was filtered as too low-count to test.


In [ ]:
res <- results(dds, contrast = c("condition", "B", "A"))
summary(res)
head(res[order(res$padj), ], 10)


In [ ]:
# a clean list of hits: significant AND at least two-fold
sig <- subset(res, padj < 0.05 & abs(log2FoldChange) > 1)
nrow(sig)


## 5. Shrink the fold changes
Use shrunken estimates whenever you rank genes or make plots.


In [ ]:
res_shrunk <- lfcShrink(dds, coef = "condition_B_vs_A", type = "apeglm")
head(res_shrunk[order(res_shrunk$padj), ], 10)


## 6. Save
This table feeds the next lessons (volcano plots, enrichment).


In [ ]:
write.csv(as.data.frame(res), "deseq2_results.csv")


### Next steps
Try a real dataset (the airway experiment, GEO GSE52778) and a design that controls for a covariate, e.g. ~ cell + dex.
